# Aula 7 — Frameworks de Qualidade de Dados

Nesta aula prática, exploramos os principais frameworks de qualidade de dados do ecossistema Python:
**Great Expectations** e **Pandera**. São ferramentas que transformam a qualidade de dados de um processo
manual para algo automatizado, proativo e rastreável.

Utilizaremos um dataset sintético de imóveis para ilustrar todos os exemplos — o mesmo cenário
da videoaula. O pipeline segue esta ordem:

1. Setup e dataset
2. Validação Imperativa vs. Declarativa
3. Great Expectations — Suite básica
4. Great Expectations — Categorias de Expectations
5. Pandera — `DataFrameSchema`
6. Pandera — `DataFrameModel` (abordagem orientada a classes)
7. Pandera — Checks customizados

## 1. Setup e Dataset

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="pandera")

import pandas as pd
import numpy as np

np.random.seed(42)

# Dataset sintético: feed diário de imóveis recebidos por uma imobiliária
# Contém intencionalmente alguns registros inválidos para acionar as validações
df_imoveis = pd.DataFrame({
    'id_imovel':       [101, 102, 103, 104, 105, 106, 107],
    'preco':           [450_000.0, 380_000.0, -1_000.0, 520_000.0, None, 600_000.0, 15_000_000.0],
    'quartos':         [3, 2, 4, 3, 5, 2, 25],   # 25 quartos é suspeito
    'area_m2':         [85.5, 72.0, 90.0, 95.5, 110.0, 68.0, 5_500.0],  # 5500 m² fora de faixa
    'cidade':          ['SP', 'RJ', 'SP', None, 'BH', 'SP', 'RJ'],
    'data_registro':   pd.to_datetime(['2025-01-10', '2025-01-10', '2025-01-10',
                                       '2025-01-10', '2025-01-10', '2025-01-10', '2025-01-10'])
})

print(df_imoveis.to_string())

   id_imovel       preco  quartos  area_m2 cidade data_registro
0        101    450000.0        3     85.5     SP    2025-01-10
1        102    380000.0        2     72.0     RJ    2025-01-10
2        103     -1000.0        4     90.0     SP    2025-01-10
3        104    520000.0        3     95.5   None    2025-01-10
4        105         NaN        5    110.0     BH    2025-01-10
5        106    600000.0        2     68.0     SP    2025-01-10
6        107  15000000.0       25   5500.0     RJ    2025-01-10


## 2. Validação Imperativa vs. Declarativa

Antes de usar os frameworks, é útil entender a diferença filosófica entre as duas abordagens.

- **Imperativa**: o desenvolvedor escreve código procedural que executa verificações explicitamente.
  Flexível, mas mistura lógica de validação com lógica de negócio.
- **Declarativa**: o desenvolvedor *declara* regras; o framework se encarrega de executar e reportar.
  As regras viram artefatos versionáveis, reutilizáveis e auto-documentáveis.

In [2]:
# ── Abordagem IMPERATIVA ──────────────────────────────────────────────────────
# Verificações escritas manualmente, espalhadas pelo código

erros = []

# Regra 1: preço não pode ser nulo ou negativo
if df_imoveis['preco'].isna().any():
    erros.append("Campo 'preco' contém valores nulos")
if (df_imoveis['preco'].dropna() <= 0).any():
    erros.append("Campo 'preco' contém valores negativos ou zero")

# Regra 2: número de quartos entre 1 e 20
fora_faixa = df_imoveis['quartos'].between(1, 20)
if not fora_faixa.all():
    erros.append(f"Campo 'quartos' fora da faixa [1, 20]: {df_imoveis.loc[~fora_faixa, 'id_imovel'].tolist()}")

# Regra 3: cidade não pode ser nula
if df_imoveis['cidade'].isna().any():
    erros.append("Campo 'cidade' contém valores nulos")

if erros:
    print("❌ Erros encontrados (abordagem imperativa):")
    for e in erros:
        print(f"  - {e}")
else:
    print("✅ Todos os dados válidos")

❌ Erros encontrados (abordagem imperativa):
  - Campo 'preco' contém valores nulos
  - Campo 'preco' contém valores negativos ou zero
  - Campo 'quartos' fora da faixa [1, 20]: [107]
  - Campo 'cidade' contém valores nulos


In [3]:
# ── Abordagem DECLARATIVA (prévia com Pandera) ────────────────────────────────
# As mesmas 3 regras, declaradas como schema — sem lógica de execução explícita

import pandera.pandas as pa
from pandera.pandas import Column, DataFrameSchema, Check
from pandera.errors import SchemaErrors

schema_declarativo = DataFrameSchema({
    'preco':   Column(float, [Check.greater_than(0)], nullable=False),
    'quartos': Column(int,   [Check.in_range(1, 20)]),
    'cidade':  Column(str,   nullable=False),
})

try:
    schema_declarativo.validate(df_imoveis[["preco", "quartos", "cidade"]], lazy=True)
    print("✅ Todos os dados válidos")
except SchemaErrors as e:
    print("❌ Erros encontrados (abordagem declarativa):")
    print(e.failure_cases[["schema_context", "column", "check", "failure_case"]].to_string(index=False))

❌ Erros encontrados (abordagem declarativa):
schema_context  column           check  failure_case
        Column   preco    not_nullable           NaN
        Column   preco greater_than(0)       -1000.0
        Column quartos in_range(1, 20)          25.0
        Column  cidade    not_nullable           NaN


> **Observação**: o resultado é o mesmo, mas a abordagem declarativa é mais concisa, legível
> e auto-documentável. As regras se tornam artefatos que podem ser versionados e compartilhados.

## 3. Great Expectations — Suite Básica

O Great Expectations (GX) é o framework de qualidade de dados mais popular do ecossistema Python.
Opera em torno de quatro conceitos: **Data Sources**, **Expectations**, **Expectation Suites**
e **Validation Definitions**.

A versão 1.0 (lançada em 2024) simplificou significativamente a API anterior (v0.x).

In [4]:
import great_expectations as gx

# Contexto efêmero: não persiste nada em disco — ideal para exemplos e testes
context = gx.get_context(mode="ephemeral")

# ── 1. Data Source e Asset ────────────────────────────────────────────────────
# Conecta o GX ao nosso DataFrame Pandas
data_source = context.data_sources.add_pandas("imoveis_ds")
data_asset  = data_source.add_dataframe_asset("imoveis_asset")
batch_def   = data_asset.add_batch_definition_whole_dataframe("batch_def")

# ── 2. Expectation Suite ──────────────────────────────────────────────────────
suite = context.suites.add(gx.ExpectationSuite(name="suite_imoveis"))

# Campos obrigatórios (não nulos)
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="preco"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="cidade"))

# Faixas de valores aceitáveis
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="preco", min_value=0, max_value=10_000_000))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="quartos", min_value=1, max_value=20))

# ── 3. Validation Definition + execução ──────────────────────────────────────
# O dataframe é passado em run() via batch_parameters
val_def = context.validation_definitions.add(
    gx.ValidationDefinition(name="val_imoveis", data=batch_def, suite=suite)
)
results = val_def.run(batch_parameters={"dataframe": df_imoveis})

# Resumo dos resultados
print(f"Validação aprovada: {results.success}\n")
for r in results.results:
    status = "✅" if r.success else "❌"
    col    = r.expectation_config.kwargs.get("column", "tabela")
    check  = r.expectation_config.type
    print(f"  {status} [{col}] {check}")

Calculating Metrics:   0%|          | 0/25 [00:00<?, ?it/s]

Validação aprovada: False

  ❌ [preco] expect_column_values_to_not_be_null
  ❌ [preco] expect_column_values_to_be_between
  ❌ [cidade] expect_column_values_to_not_be_null
  ❌ [quartos] expect_column_values_to_be_between


## 4. Great Expectations — Categorias de Expectations

O GX organiza suas mais de 300 expectations em três categorias principais:

| Categoria | O que verifica | Exemplo |
|-----------|---------------|---------|
| **Column Map** | Cada linha individualmente | `values_to_match_regex` |
| **Column Aggregate** | Estatísticas da coluna | `mean_to_be_between` |
| **Table** | Propriedades da tabela | `row_count_to_be_between` |

In [ ]:
# Novo contexto limpo para demonstrar as categorias
ctx2 = gx.get_context(mode='ephemeral')

ds2    = ctx2.data_sources.add_pandas('ds2')
asset2 = ds2.add_dataframe_asset('asset2')
bd2    = asset2.add_batch_definition_whole_dataframe('bd2')

suite2 = ctx2.suites.add(gx.ExpectationSuite(name='suite_categorias'))

# ── Column MAP Expectations (verificam linha a linha) ────────────────────────
# Todos os valores de 'cidade' devem pertencer ao conjunto permitido
# Os valores nulos não são reportados por esta expectativa, 
# mas poderiam ser capturados por outra expectativa de não nulidade.
suite2.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(
    column='cidade', value_set=['SP', 'RJ', 'MG', 'BH', 'RS']))

# Ids devem ser inteiros positivos (regex simples)
suite2.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(
    column='id_imovel'))

# ── Column AGGREGATE Expectations (estatísticas da coluna) ───────────────────
# Média de preços deve estar em faixa histórica esperada
suite2.add_expectation(gx.expectations.ExpectColumnMeanToBeBetween(
    column='preco', min_value=100_000, max_value=2_000_000))

# id_imovel deve ser praticamente único (proporção >= 99%)
suite2.add_expectation(gx.expectations.ExpectColumnProportionOfUniqueValuesToBeBetween(
    column='id_imovel', min_value=0.99, max_value=1.0))

# ── TABLE Expectations (sobre a tabela inteira) ───────────────────────────────
# Volume mínimo e máximo de registros esperado num feed diário
suite2.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(
    min_value=1, max_value=10_000))

# Colunas obrigatórias devem existir na ordem correta
suite2.add_expectation(gx.expectations.ExpectTableColumnsToMatchOrderedList(
    column_list=['id_imovel', 'preco', 'quartos', 'area_m2', 'cidade', 'data_registro']))

# ── Execução ──────────────────────────────────────────────────────────────────
val2    = ctx2.validation_definitions.add(
    gx.ValidationDefinition(name='val_categorias', data=bd2, suite=suite2)
)
res2    = val2.run(batch_parameters={'dataframe': df_imoveis})

print(f"Validação aprovada: {res2.success}\n")
for r in res2.results:
    status = '✅' if r.success else '❌'
    col    = r.expectation_config.kwargs.get('column', 'tabela')
    check  = r.expectation_config.type
    print(f"  {status} [{col}] {check}")

Calculating Metrics:   0%|          | 0/21 [00:00<?, ?it/s]

Validação aprovada: False

  ✅ [cidade] expect_column_values_to_be_in_set
  ✅ [id_imovel] expect_column_values_to_not_be_null
  ✅ [id_imovel] expect_column_proportion_of_unique_values_to_be_between
  ❌ [preco] expect_column_mean_to_be_between
  ✅ [tabela] expect_table_row_count_to_be_between
  ✅ [tabela] expect_table_columns_to_match_ordered_list


In [ ]:
# ── Data Docs: documentação automática ───────────────────────────────────────
# O GX gera um site HTML com todas as suites e resultados de validação.
# Em ambientes locais com um contexto persistente (não ephemeral), execute:
#
# context.build_data_docs()
# context.open_data_docs()   # Abre no navegador
#
# Os docs ficam em: great_expectations/uncommitted/data_docs/local_site/

print("Data Docs: documentação automática gerada pelo GX após cada validação.")
print("Disponível com contexto persistente (mode='file' ou 'cloud').")

Data Docs: documentação automática gerada pelo GX após cada validação.
Disponível com contexto persistente (mode='file' ou 'cloud').


## 5. Pandera — `DataFrameSchema`

O Pandera é um framework de validação fortemente integrado ao ecossistema Python moderno.
O `DataFrameSchema` é a abordagem clássica: um dicionário mapeando colunas para suas regras.

In [7]:
import pandera.pandas as pa
from pandera.pandas import Column, DataFrameSchema, Check
from pandera.errors import SchemaErrors

# Schema declarativo: descreve a estrutura e as regras de qualidade do DataFrame
schema_imoveis = DataFrameSchema({
    "preco": Column(float, [
        Check.greater_than(0),
        Check.less_than(10_000_000),
    ], nullable=False),

    "quartos": Column(int, [
        Check.in_range(1, 20)
    ]),

    "area_m2": Column(float, [
        Check.greater_than(10),
        Check.less_than(5_000)
    ]),

    "cidade": Column(str, nullable=False)
})

# lazy=True coleta TODOS os erros antes de lançar a exceção
# (sem lazy, para no primeiro erro)
try:
    schema_imoveis.validate(df_imoveis, lazy=True)
    print("✅ Dados válidos!")
except SchemaErrors as e:
    print("❌ Erros encontrados:")
    print(e.failure_cases[["schema_context", "column", "check", "failure_case"]].to_string(index=False))

❌ Erros encontrados:
schema_context  column               check  failure_case
        Column   preco        not_nullable           NaN
        Column   preco     greater_than(0)       -1000.0
        Column   preco less_than(10000000)    15000000.0
        Column quartos     in_range(1, 20)          25.0
        Column area_m2     less_than(5000)        5500.0
        Column  cidade        not_nullable           NaN


## 6. Pandera — `DataFrameModel` (abordagem orientada a classes)

O `DataFrameModel` é a abordagem mais moderna do Pandera: define o schema como uma **classe Python**
com anotações de tipo, similar ao Pydantic para dados tabulares. Integra com `mypy`/`pyright`
e permite usar o schema como *type hint* em funções, validando automaticamente inputs e outputs.

Esta abordagem é destaque na seção SAIBA MAIS como a escolha ideal para equipes que valorizam
*type safety* e documentação via tipos.

In [8]:
import pandera.pandas as pa
from pandera.typing import DataFrame, Series
from pandera.errors import SchemaErrors
from typing import Optional

# Schema como classe — sintaxe declarativa orientada a tipos
class ImoveisSchema(pa.DataFrameModel):
    id_imovel:      Series[int]   = pa.Field(unique=True, gt=0)
    preco:          Series[float] = pa.Field(gt=0, lt=10_000_000)
    quartos:        Series[int]   = pa.Field(ge=1, le=20)
    area_m2:        Series[float] = pa.Field(gt=10, lt=5_000)
    cidade:         Series[str]   = pa.Field(isin=["SP", "RJ", "MG", "BH", "RS"])
    data_registro:  Series[pa.DateTime] = pa.Field()

    class Config:
        coerce = True   # tenta converter tipos automaticamente (ex: int → float)
        strict = False  # permite colunas extras no DataFrame


# @pa.check_types valida automaticamente o input e output da função
@pa.check_types
def enriquecer_imoveis(df: DataFrame[ImoveisSchema]) -> DataFrame[ImoveisSchema]:
    """Adiciona o campo preco_por_m2 ao DataFrame de imóveis."""
    df = df.copy()
    df["preco_por_m2"] = df["preco"] / df["area_m2"]
    return df


# Dataset limpo para demonstrar o caminho feliz
df_limpo = pd.DataFrame({
    "id_imovel":      [101, 102, 103],
    "preco":          [450_000.0, 380_000.0, 520_000.0],
    "quartos":        [3, 2, 3],
    "area_m2":        [85.5, 72.0, 95.5],
    "cidade":         ["SP", "RJ", "SP"],
    "data_registro":  pd.to_datetime(["2025-01-10", "2025-01-10", "2025-01-10"]),
})

resultado = enriquecer_imoveis(df_limpo)
print(resultado[["id_imovel", "preco", "area_m2", "preco_por_m2"]].to_string(index=False))

 id_imovel    preco  area_m2  preco_por_m2
       101 450000.0     85.5   5263.157895
       102 380000.0     72.0   5277.777778
       103 520000.0     95.5   5445.026178


In [9]:
# Validação explícita com DataFrameModel
# Equivalente ao schema_imoveis.validate() do exemplo anterior
try:
    ImoveisSchema.validate(df_imoveis, lazy=True)
    print("✅ Dados válidos!")
except SchemaErrors as e:
    print("❌ Erros encontrados via DataFrameModel:")
    print(e.failure_cases[["schema_context", "column", "check", "failure_case"]].to_string(index=False))

❌ Erros encontrados via DataFrameModel:
schema_context  column                     check  failure_case
        Column   preco              not_nullable           NaN
        Column   preco           greater_than(0)       -1000.0
        Column   preco       less_than(10000000)    15000000.0
        Column quartos less_than_or_equal_to(20)          25.0
        Column area_m2           less_than(5000)        5500.0
        Column  cidade              not_nullable           NaN


## 7. Pandera — Checks Customizados

Além das validações de schema básicas, o Pandera permite:
- **Checks por lambda**: funções arbitrárias sobre uma coluna
- **Checks de nível agregado**: verificam estatísticas (`mean`, `std`, etc.)
- **Hipóteses estatísticas**: testes formais entre grupos (ex: teste t de Student)

Esses checks são especialmente úteis para detectar *data drift* e verificar relações
de negócio que não se expressam como simples faixas de valores.

In [10]:
import pandera.pandas as pa
from pandera.pandas import Column, DataFrameSchema, Check
from pandera.errors import SchemaErrors
import pandas as pd

# ── Check por lambda (linha a linha) ─────────────────────────────────────────
# element_wise=True → lambda recebe cada valor individualmente (scalar)
# Regra de negócio: preço deve ser divisível por 1000 (ex: política da empresa)
check_multiplo_de_mil = Check(
    lambda x: pd.isna(x) or x % 1_000 == 0,
    element_wise=True,
    error="Preco deve ser multiplo de R$1.000"
)

# ── Check agregado (sobre a coluna inteira) ───────────────────────────────────
# element_wise=False (padrão) → lambda recebe a Series completa
# Média histórica esperada entre R$200k e R$800k
check_media_historica = Check(
    lambda s: 200_000 <= s.mean() <= 800_000,
    element_wise=False,
    error="Media de precos fora da faixa historica [200k, 800k]"
)

schema_custom = DataFrameSchema({
    "preco": Column(float, [
        Check.greater_than(0),
        check_media_historica,
        check_multiplo_de_mil,
    ], nullable=False),
    "quartos": Column(int, [Check.in_range(1, 20)]),
})

try:
    schema_custom.validate(df_limpo, lazy=True)
    print("✅ Todos os checks passaram")
except SchemaErrors as e:
    print("❌ Falhas nos checks customizados:")
    print(e.failure_cases[["column", "check", "failure_case"]].to_string(index=False))

✅ Todos os checks passaram


In [11]:
# ── Hipótese estatística: teste t entre grupos ────────────────────────────────
# Expectativa de negócio: imóveis grandes (3+ quartos) têm preço médio MAIOR
# que imóveis pequenos. Verificado formalmente com teste t de Student (alpha=0.05).
#
# Atenção: Hypothesis deve ser declarada como check de Column (não de DataFrameSchema).
# O groupby referencia outra coluna do DataFrame que define os grupos.

from pandera.pandas import Hypothesis
from pandera.errors import SchemaError

schema_hipotese = pa.DataFrameSchema({
    # A hipótese é declarada na coluna alvo (preco)
    "preco": pa.Column(float, [
        pa.Check.greater_than(0),
        Hypothesis.two_sample_ttest(
            sample1="grande",     # grupo 1: imóveis com 3+ quartos
            sample2="pequeno",    # grupo 2: imóveis com 1-2 quartos
            groupby="porte",      # coluna que define os grupos
            relationship="greater_than",
            alpha=0.05,
            error="Imoveis grandes devem ter preco medio estatisticamente maior"
        )
    ]),
    "porte": pa.Column(str),  # coluna categórica que define o grupo
})

# Dataset com contraste claro entre os grupos
df_grupos = pd.DataFrame({
    "preco": [500_000.0, 520_000.0, 480_000.0, 200_000.0, 220_000.0, 210_000.0],
    "porte": ["grande", "grande", "grande", "pequeno", "pequeno", "pequeno"],
})

try:
    schema_hipotese.validate(df_grupos)
    print("✅ Hipótese confirmada: imóveis grandes têm preço médio maior (p < 0.05)")
except SchemaError as e:
    print(f"❌ Hipótese rejeitada: {e}")

✅ Hipótese confirmada: imóveis grandes têm preço médio maior (p < 0.05)
